# Bifurcations: equilibria, a Hopf point and the periodic branch

The first notebook changed `iapp` by hand and saw the Morris-Lecar cell go
from rest to firing. AUTO, XPPAUT's continuation code, follows the
equilibria as `iapp` changes and finds where their stability changes: the
bifurcation points. This notebook computes the branch of equilibria, finds
its Hopf points and follows the branch of periodic orbits that starts at
one of them.

It uses the model's `hopf` parameter set, stored in the `.ode` file.

In [ ]:
import matplotlib.pyplot as plt
import xpplab as xs

m = xs.load(xs.example("lecar.ode"))
m.apply_set("hopf")
m.parameters

## A branch of equilibria

`m.auto.equilibria("iapp", parmax=0.6)` moves the initial values onto an
equilibrium and follows it for `iapp` up to 0.6. The result is a `Diagram`.

In [ ]:
d = m.auto.equilibria("iapp", parmax=0.6)
d

In [ ]:
d.plot("V");

Red is a stable equilibrium and dashed black an unstable one. The orange
dots are the special points AUTO found.

In [ ]:
d.special()

There are two Hopf points (`HB`): the equilibrium is stable for small
`iapp`, unstable between the two and stable again above the second.

In [ ]:
hb = d.first("HB")
hb.value, hb.stable, hb.eigenvalues

At the Hopf point the eigenvalues are a pure imaginary pair (the real part
is zero up to the accuracy of the computation). The imaginary part, about
0.43, gives the frequency of the small oscillations that are born there.

## The periodic branch from the Hopf point

`m.auto.continue_from(hb, "periodic")` follows the periodic orbits that
leave the Hopf point. `nmax` limits the number of steps and `npr` how often
a point is kept with its whole orbit.

In [ ]:
orbits = m.auto.continue_from(hb, "periodic", nmax=45, npr=5)
orbits.special()

In [ ]:
m.auto.diagram.plot("V");

The green curves are the largest and smallest `V` of stable periodic
orbits, the dashed blue curve those of unstable ones. The branch leaves the
Hopf point towards *smaller* `iapp`: the unstable orbits turn around at a
fold (`LP`) and become stable. This is a subcritical Hopf bifurcation. Between
the fold and the Hopf point a stable rest state and a stable oscillation
exist together, and which one you see depends on where you start.

In [ ]:
fold = orbits.first("LP")
fold.value, fold.period

In [ ]:
print("stable rest state for iapp <", round(hb.value, 3))
print("stable oscillation from iapp =", round(fold.value, 3), "up to about 0.47")

You can check the diagram by simulation. At `iapp = 0.3` the equilibrium is
unstable, so a run from near it ends on the stable oscillation.

In [ ]:
m.parameters["iapp"] = 0.3
sol = m.run(total=300)
sol.plot("V");

In [ ]:
late = sol["V"][sol.t > 200]
late.min(), late.max()

These are the smallest and largest `V` on the oscillation at `iapp = 0.3`,
which is where the green branch of the diagram passes at that value.
`m.auto.orbit(label)` returns the orbit stored at a labelled point as a
trajectory.

## Where next

- `03_control.ipynb` uses `iapp` as a control input and stabilises the
  unstable equilibrium that this diagram shows.
- [Bifurcations with AUTO](https://cesarperdomodiaz.github.io/xpplab/python/auto/)
  lists every continuation a point allows.